# Isotools differential splicing analysis

## Setup

In [7]:
from isotools import Transcriptome
import matplotlib.pyplot as plt
import pandas as pd

In [8]:
path='/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output'
isoseq=Transcriptome.load(f'{path}/A_najas_isotools_filtered.pkl')

In [9]:
import numpy as np
import sys
import io
from contextlib import redirect_stderr
from statsmodels.stats.multitest import multipletests

# Define parameters
outpath = "/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS"  
types_of_interest = ['ES', 'ME', '5AS', '3AS', 'IR']  # Ignore TSS and PAS

## I1 vs I2

In [10]:
group1_name = 'I1'
group2_name = 'I2'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I1 vs I2
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1792 problematic genes out of 15472
Saved problematic genes to 'I1_I2_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 5247 events across 2272 genes
Skipped 1792 problematic genes

RESULTS:
  2556 differential splice sites in 1135 genes

Significant events by type:
  ES: 236
  ME: 66
  5AS: 418
  3AS: 370
  IR: 1466

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I1_I2/I1_I2_differential_events.csv'
Updated problematic genes saved to 'I1_I2_problematic_genes_updated.txt'


## I2 vs I3

In [11]:
group1_name = 'I2'
group2_name = 'I3'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I2 vs I3
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1814 problematic genes out of 15472
Saved problematic genes to 'I2_I3_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 4850 events across 2125 genes
Skipped 1814 problematic genes

RESULTS:
  2660 differential splice sites in 1184 genes

Significant events by type:
  ES: 273
  ME: 79
  5AS: 452
  3AS: 395
  IR: 1461

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I2_I3/I2_I3_differential_events.csv'
Updated problematic genes saved to 'I2_I3_problematic_genes_updated.txt'


## I3 vs I4

In [12]:
group1_name = 'I3'
group2_name = 'I4'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I3 vs I4
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1826 problematic genes out of 15472
Saved problematic genes to 'I3_I4_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 4112 events across 1982 genes
Skipped 1826 problematic genes

RESULTS:
  2201 differential splice sites in 1128 genes

Significant events by type:
  ES: 278
  ME: 82
  5AS: 414
  3AS: 352
  IR: 1075

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I3_I4/I3_I4_differential_events.csv'
Updated problematic genes saved to 'I3_I4_problematic_genes_updated.txt'


## I4 vs I5F

In [13]:
group1_name = 'I4'
group2_name = 'I5F'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I4 vs I5F
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1843 problematic genes out of 15472
Saved problematic genes to 'I4_I5F_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 3752 events across 1892 genes
Skipped 1843 problematic genes

RESULTS:
  2457 differential splice sites in 1239 genes

Significant events by type:
  ES: 317
  ME: 72
  5AS: 418
  3AS: 339
  IR: 1311

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I4_I5F/I4_I5F_differential_events.csv'
Updated problematic genes saved to 'I4_I5F_problematic_genes_updated.txt'


## I4 vs I5M

In [14]:
group1_name = 'I4'
group2_name = 'I5M'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I4 vs I5M
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1843 problematic genes out of 15472
Saved problematic genes to 'I4_I5M_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 4278 events across 2097 genes
Skipped 1843 problematic genes

RESULTS:
  2651 differential splice sites in 1311 genes

Significant events by type:
  ES: 334
  ME: 95
  5AS: 486
  3AS: 350
  IR: 1386

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I4_I5M/I4_I5M_differential_events.csv'
Updated problematic genes saved to 'I4_I5M_problematic_genes_updated.txt'


## I5F vs I5M

In [16]:
group1_name = 'I5F'
group2_name = 'I5M'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I5F vs I5M
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1848 problematic genes out of 15472
Saved problematic genes to 'I5F_I5M_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 3792 events across 1852 genes
Skipped 1848 problematic genes

RESULTS:
  2433 differential splice sites in 1179 genes

Significant events by type:
  ES: 292
  ME: 80
  5AS: 437
  3AS: 375
  IR: 1249

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I5F_I5M/I5F_I5M_differential_events.csv'
Updated problematic genes saved to 'I5F_I5M_problematic_genes_updated.txt'


## I5F vs Fad

In [17]:
group1_name = 'I5F'
group2_name = 'Fad'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I5F vs Fad
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1837 problematic genes out of 15472
Saved problematic genes to 'I5F_Fad_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 3547 events across 1808 genes
Skipped 1837 problematic genes

RESULTS:
  2358 differential splice sites in 1177 genes

Significant events by type:
  ES: 292
  ME: 82
  5AS: 447
  3AS: 313
  IR: 1224

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I5F_Fad/I5F_Fad_differential_events.csv'
Updated problematic genes saved to 'I5F_Fad_problematic_genes_updated.txt'


## I5M vs Mad

In [18]:
group1_name = 'I5M'
group2_name = 'Mad'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing I5M vs Mad
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1854 problematic genes out of 15472
Saved problematic genes to 'I5M_Mad_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 4273 events across 2116 genes
Skipped 1854 problematic genes

RESULTS:
  2420 differential splice sites in 1217 genes

Significant events by type:
  ES: 322
  ME: 90
  5AS: 471
  3AS: 330
  IR: 1207

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/I5M_Mad/I5M_Mad_differential_events.csv'
Updated problematic genes saved to 'I5M_Mad_problematic_genes_updated.txt'


## Fad vs Mad

In [19]:
group1_name = 'Fad'
group2_name = 'Mad'
group_dict = {group1_name: [group1_name], group2_name: [group2_name]}
pair = f'{group1_name}_{group2_name}'

print(f"Comparing {group1_name} vs {group2_name}")
print(f"Event types: {types_of_interest}")

# First pass: identify problematic genes
print("\n" + "="*60)
print("IDENTIFYING PROBLEMATIC GENES (SILENT MODE)")
print("="*60)

problematic_genes = []
total_genes = sum(1 for _ in isoseq.iter_genes())

# Redirect stderr to suppress error messages
stderr_buffer = io.StringIO()

for gene in isoseq.iter_genes():
    # Redirect stderr to ignore the error messages
    with redirect_stderr(stderr_buffer):
        try:
            # Try to run test on this gene
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
        except Exception:
            # Any exception means this gene is problematic
            problematic_genes.append(gene.id)
            continue
        
        # If we get results, check if they're valid
        if result is not None and len(result) > 0:
            # Check if pvalue column exists and is numeric
            if 'pvalue' in result.columns:
                if not pd.api.types.is_numeric_dtype(result['pvalue']):
                    problematic_genes.append(gene.id)
            else:
                problematic_genes.append(gene.id)

print(f"\nFound {len(problematic_genes)} problematic genes out of {total_genes}")

# Save problematic genes list
with open(f'{outpath}/{pair}/{pair}_problematic_genes.txt', 'w') as f:
    for gene_id in problematic_genes:
        f.write(f"{gene_id}\n")
print(f"Saved problematic genes to '{pair}_problematic_genes.txt'")

# Now run the actual test skipping problematic genes (also silent)
print("\n" + "="*60)
print("RUNNING DIFFERENTIAL SPLICING TEST")
print("="*60)

all_results = []
genes_with_events = 0
skipped = 0

for gene in isoseq.iter_genes():
    if gene.id in problematic_genes:
        skipped += 1
        continue
    
    # Redirect stderr again
    with redirect_stderr(stderr_buffer):
        try:
            result = isoseq.altsplice_test(
                group_dict,
                types=types_of_interest,
                min_total=16,
                min_alt_fraction=0.125,
                test="binom_lr",
                region=f"{gene.chrom}:{gene.start}-{gene.end}"
            )
            
            if result is not None and len(result) > 0:
                all_results.append(result)
                genes_with_events += 1
                
        except Exception:
            # Add to problematic if it fails
            problematic_genes.append(gene.id)
            skipped += 1

# Combine results
if all_results:
    diff_splice = pd.concat(all_results, ignore_index=True)
    print(f"\nFound {len(diff_splice)} events across {genes_with_events} genes")
    print(f"Skipped {skipped} problematic genes")
    
    # Ensure pvalues are numeric
    diff_splice['pvalue'] = pd.to_numeric(diff_splice['pvalue'], errors='coerce')
    diff_splice = diff_splice.dropna(subset=['pvalue'])
    
    # Calculate adjusted p-values manually
    diff_splice['padj'] = multipletests(diff_splice['pvalue'], method='fdr_bh')[1]
    
    # Sort and get significant events
    diff_splice = diff_splice.sort_values('pvalue').reset_index(drop=True)
    sig = diff_splice['padj'] < 0.05
    
    n_genes = len(diff_splice.loc[sig, "gene"].unique())
    print(f"\nRESULTS:")
    print(f"  {sum(sig)} differential splice sites in {n_genes} genes")
    
    # Show by type
    if sum(sig) > 0:
        print(f"\nSignificant events by type:")
        sig_by_type = diff_splice.loc[sig, 'splice_type'].value_counts()
        for event_type in types_of_interest:
            if event_type in sig_by_type:
                print(f"  {event_type}: {sig_by_type[event_type]}")
    
    # Save results
    output_file = f'{outpath}/{pair}/{pair}_differential_events.csv'
    diff_splice.to_csv(output_file, index=False)
    print(f"\nResults saved to '{output_file}'")
    
    # Save updated problematic genes
    with open(f'{outpath}/{pair}/{pair}_problematic_genes_updated.txt', 'w') as f:
        for gene_id in sorted(set(problematic_genes)):
            f.write(f"{gene_id}\n")
    print(f"Updated problematic genes saved to '{pair}_problematic_genes_updated.txt'")
    
else:
    print("No events found")

Comparing Fad vs Mad
Event types: ['ES', 'ME', '5AS', '3AS', 'IR']

IDENTIFYING PROBLEMATIC GENES (SILENT MODE)

Found 1846 problematic genes out of 15472
Saved problematic genes to 'Fad_Mad_problematic_genes.txt'

RUNNING DIFFERENTIAL SPLICING TEST

Found 3916 events across 1973 genes
Skipped 1846 problematic genes

RESULTS:
  2492 differential splice sites in 1256 genes

Significant events by type:
  ES: 339
  ME: 89
  5AS: 472
  3AS: 344
  IR: 1248

Results saved to '/home/ingomue/1.Work/1.Projects/2025_Aquarius/15.isotools_final/01.output/02.DS/Fad_Mad/Fad_Mad_differential_events.csv'
Updated problematic genes saved to 'Fad_Mad_problematic_genes_updated.txt'
